[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-3-ai-agents/adv-03b-skills-and-progressive-disclosure.ipynb)

# Advanced Discussion 3 (continued) — skills and progressive disclosure

Token cost of loading every skill versus advertising descriptions, and how well different advertised texts separate 24 skills (embedding proxy).

**Optional advanced-discussion lab** for AI Agents and Agentic AI with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Downloads a small embedding model (about 90 MB); runs in about a minute.

In [ ]:
%pip install -q sentence-transformers tiktoken numpy

In [ ]:
import numpy as np, math
from sentence_transformers import SentenceTransformer
rng = np.random.RandomState(0)

## 1. a catalogue of 24 skills: each has a short vague label, a rich description (what it does + when to use it), a body, and two realistic user requests

In [ ]:
S = {
 "pdf-extract":     ("PDF helper", "Extract text, tables and form fields from existing PDF files. Use when the user has a PDF and wants its contents read, searched or converted to data.", ["pull the tables out of this annual report pdf", "what does the attached contract pdf say about termination"]),
 "pdf-create":      ("PDF helper", "Generate new PDF documents from templates or data: invoices, reports, certificates. Use when the user wants a PDF produced, not read.", ["make a pdf invoice for order 4471", "turn this summary into a one page pdf report"]),
 "sql-write":       ("database helper", "Write SQL queries from a plain-language question, given the table schema. Use for questions like 'how many', 'top ten', 'average per month'.", ["how many orders were placed last month by region", "give me the ten customers with the highest spend"]),
 "sql-optimise":    ("database helper", "Diagnose and speed up slow SQL queries: read the query plan, suggest indexes and rewrites. Use when a query already exists but is slow.", ["this query takes four minutes can you speed it up", "which index would help this join on orders and customers"]),
 "chart-make":      ("visualisation helper", "Create charts and plots from tabular data with clear labels. Use when the user asks for a graph, plot or visual of numbers.", ["plot monthly revenue as a line chart", "show me a bar chart of returns by category"]),
 "stats-test":      ("analysis helper", "Run statistical tests and report effect sizes and confidence intervals. Use when comparing groups or asking if a difference is significant.", ["is the conversion rate difference between the two variants significant", "compare average delivery time before and after the change"]),
 "email-draft":     ("writing helper", "Draft professional emails in the user's tone from a short brief. Use when the user needs a message written to a person.", ["write an email to the carrier asking about the delayed shipment", "draft a polite reply declining the meeting"]),
 "meeting-notes":   ("writing helper", "Turn a meeting transcript into decisions, action items with owners, and open questions. Use after a meeting when a transcript or notes exist.", ["here is the call transcript, list the action items", "summarise the decisions from yesterday's planning meeting"]),
 "translate":       ("language helper", "Translate text between languages preserving tone and formatting, and flag ambiguous phrases. Use when text is in another language.", ["translate this customer message from german to english", "convert the product description into french"]),
 "code-review":     ("coding helper", "Review a code change for bugs, security issues and style, with specific line-level comments. Use when the user shares a diff or pull request.", ["review this pull request for bugs", "does this function have any security problems"]),
 "code-test":       ("coding helper", "Write unit tests for existing functions, including edge cases, using the project's test framework. Use when code exists but lacks tests.", ["write tests for the parse_date function", "add edge case tests to this module"]),
 "code-debug":      ("coding helper", "Find the cause of an error from a stack trace and logs, and propose a minimal fix. Use when something crashes or behaves wrongly.", ["I get a keyerror on line 40, what is wrong", "the script hangs after the second request, why"]),
 "web-research":    ("research helper", "Search the web, read sources and produce a cited summary. Use when the answer depends on current public information.", ["what are the current export rules for lithium batteries", "find recent news about the port strike"]),
 "doc-qa":          ("research helper", "Answer questions from the company's internal documents with citations to the source passage. Use for policy, procedure and product questions.", ["what is our refund policy for damaged goods", "which approval is needed for purchases over the limit"]),
 "calendar":        ("scheduling helper", "Find free slots, propose meeting times and create calendar events. Use when scheduling or rescheduling.", ["find a time next week for a one hour review with dana", "move tomorrow's standup to the afternoon"]),
 "expense-report":  ("finance helper", "Categorise receipts and build an expense report following company rules. Use when the user submits receipts or asks for a claim.", ["file these five receipts as an expense claim", "which of these purchases is not reimbursable"]),
 "invoice-check":   ("finance helper", "Match invoices against purchase orders and flag mismatches in amount or quantity. Use for accounts payable checks.", ["does this invoice match purchase order 7712", "flag any invoices whose totals differ from the order"]),
 "data-clean":      ("data helper", "Clean messy tabular data: fix types, remove duplicates, standardise dates and missing values. Use before analysis when data looks dirty.", ["this spreadsheet has inconsistent dates and duplicate rows", "standardise the country column"]),
 "forecast":        ("analysis helper", "Build a baseline time-series forecast and evaluate it honestly against a naive benchmark. Use for demand or sales prediction.", ["forecast next quarter's demand for this product", "predict weekly volumes for the warehouse"]),
 "ticket-triage":   ("support helper", "Classify incoming support tickets by topic and urgency and route them. Use for new tickets awaiting assignment.", ["sort these new tickets by urgency", "which team should get this complaint about billing"]),
 "policy-check":    ("compliance helper", "Check a document or action against company policy and regulations and cite the rule. Use when compliance is in question.", ["is this contract clause allowed under our data policy", "does this marketing claim break any advertising rules"]),
 "slides":          ("presentation helper", "Build a slide deck from an outline with consistent layout. Use when the user wants a presentation.", ["turn this outline into a ten slide deck", "make slides summarising the quarterly results"]),
 "image-describe":  ("vision helper", "Describe images and read text in screenshots, photos and scans. Use when the user attaches an image.", ["what does this screenshot of the dashboard show", "read the text on this photo of a label"]),
 "meeting-book":    ("scheduling helper", "Book meeting rooms and video links for confirmed events. Use after a time is agreed and a room is needed.", ["reserve the large room for the planning session", "add a video link to the thursday meeting"]),
}
names = list(S)
import tiktoken
_tk = tiktoken.get_encoding('cl100k_base')
BODY_TOKENS = 1500; DESC_TOKENS_RICH = int(round(np.mean([len(_tk.encode(S[n][1])) for n in S]))); DESC_TOKENS_VAGUE = int(round(np.mean([len(_tk.encode(S[n][0])) for n in S])))
print('measured description sizes: rich %d tokens, vague %d tokens (skill bodies assumed %d tokens each)' % (DESC_TOKENS_RICH, DESC_TOKENS_VAGUE, BODY_TOKENS))
queries = [(q, n) for n in names for q in S[n][2]]
print("%d skills, %d user requests" % (len(names), len(queries)))
enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
Q = enc.encode([q for q, _ in queries], normalize_embeddings=True)
def evaluate(texts, subset):
    idx = [i for i, (q, n) in enumerate(queries) if n in subset]; E = enc.encode([texts[n] for n in subset], normalize_embeddings=True)
    sc = Q[idx] @ E.T; top1 = top3 = 0
    for r, i in enumerate(idx):
        order = np.argsort(-sc[r]); truth = list(subset).index(queries[i][1]); top1 += order[0] == truth; top3 += truth in order[:3]
    return top1 / len(idx), top3 / len(idx)
variants = {"name only (e.g. 'sql-write')": {n: n.replace("-", " ") for n in names},
            "vague label ('database helper')": {n: S[n][0] for n in names},
            "rich description (what + when to use)": {n: S[n][1] for n in names}}
print("\nchoosing the right skill by matching the request to the skill's advertised text (embedding top-k), 24 skills")
print("advertised text                            top-1   top-3")
for label, tx in variants.items():
    a, b = evaluate(tx, names); print("%-42s %.2f    %.2f" % (label, a, b))
print("\nrich descriptions, by size of the catalogue (random subsets of skills, 30 draws):")
for k in (6, 12, 24):
    res = [evaluate(variants["rich description (what + when to use)"], list(np.random.RandomState(s).permutation(names)[:k])) if k < 24 else evaluate(variants["rich description (what + when to use)"], names) for s in range(30 if k < 24 else 1)]
    print("  %2d skills: top-1 %.2f  top-3 %.2f" % (k, np.mean([r[0] for r in res]), np.mean([r[1] for r in res])))
a, b = evaluate(variants["rich description (what + when to use)"], names)
# where does it go wrong?
E = enc.encode([S[n][1] for n in names], normalize_embeddings=True); sc = Q @ E.T
wrong = [(q, n, names[int(np.argmax(sc[i]))]) for i, (q, n) in enumerate(queries) if names[int(np.argmax(sc[i]))] != n]
print("\nconfusions with rich descriptions (%d of %d):" % (len(wrong), len(queries)))
for q, n, g in wrong: print("  '%s' -> chose %s, wanted %s" % (q, g, n))

## 2. the token bill: load everything vs advertise + load on demand

In [ ]:
print("\ntokens in context per request, %d skills (body %d tokens, rich description %d tokens):" % (len(names), BODY_TOKENS, DESC_TOKENS_RICH))
load_all = len(names) * BODY_TOKENS; progressive = len(names) * DESC_TOKENS_RICH + BODY_TOKENS
print("  load every skill body up front: %6d tokens" % load_all)
print("  advertise descriptions, load the chosen body: %6d tokens (%.0f%% of load-all)" % (progressive, 100 * progressive / load_all))
print("  wrong choice then a second load (about %.0f%% of requests at top-1 %.2f): expected %.0f tokens" % (100 * (1 - a), a, len(names) * DESC_TOKENS_RICH + BODY_TOKENS * (1 + (1 - a))))
print("scaling: 100 skills -> load-all %d tokens vs progressive %d; 500 skills -> %d vs %d" % (100 * BODY_TOKENS, 100 * DESC_TOKENS_RICH + BODY_TOKENS, 500 * BODY_TOKENS, 500 * DESC_TOKENS_RICH + BODY_TOKENS))
print("the advertised descriptions themselves grow linearly: 500 skills x %d = %d tokens on every request, so large catalogues need a retrieval step in front" % (DESC_TOKENS_RICH, 500 * DESC_TOKENS_RICH))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.